# M3L2 E02 - RAG mini: el pipeline completo

## Que vamos a ver

Este notebook muestra la diferencia entre un script RAG mezclado y un pipeline RAG modular
construido con LangChain.

## Por que este es el ejercicio clave de la lecture

La lecture describe este problema en la Seccion 15.1:

> Script legacy: carga documentos en cada consulta, mezcla ingestion y consulta,
> no separa retriever, prompt hardcodeado, LLM hardcodeado, dificil testing, dificil tracing.

Y la solucion en 15.2:

> Separar dos fases: Ingestion (docs → chunks → embeddings → vector store)
> y Consulta (query → retriever → prompt → llm → answer).

## Datos en memoria

Para evitar dependencias de archivos externos, usamos textos hardcodeados
que FAISS indexa en memoria. En produccion estos textos vendrian de documentos reales.

## Este notebook necesita API key de OpenAI y FAISS

Si FAISS no esta instalado, ejecuta la primer celda.


In [ ]:
# Instalar dependencias si no estan disponibles
# Descomenta y ejecuta solo si necesitas instalar
# !pip install faiss-cpu langchain langchain-openai langchain-community

import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


## Mapa de conceptos

| Concepto de la lecture | Rol en el pipeline | Componente LangChain |
|---|---|---|
| Embeddings | Convierte texto en vectores | `OpenAIEmbeddings()` |
| Vector Store | Almacena y busca por similitud | `FAISS.from_texts()` |
| Retriever | Interfaz de busqueda | `vectorstore.as_retriever()` |
| PromptTemplate | Estructura el prompt | `ChatPromptTemplate.from_messages()` |
| LLM | Genera la respuesta | `ChatOpenAI()` |
| LCEL | Conecta todo | `{...} \| prompt \| llm \| parser` |

Pipeline completo:

```text
INGESTION (una sola vez):
Textos → OpenAIEmbeddings → FAISS Vector Store

CONSULTA (por cada pregunta):
Pregunta
  |
  v
Retriever  →  documentos relevantes
  |                  |
  +------ context ---+
         |
         v
    PromptTemplate
         |
         v
       ChatOpenAI
         |
         v
    StrOutputParser
         |
         v
    Respuesta final
```


## Dos fases del pipeline RAG (Lecture M3L2 - Seccion 15.2)

La lecture M3L2 propone separar el pipeline en dos fases distintas:

```text
FASE 1: INGESTION (se hace una sola vez, o cuando cambian los documentos)
------------------------------------------------------------------
Documentos
    |
    v
Document Loader  (carga archivos, PDFs, webs, etc.)
    |
    v
Text Splitter    (divide en chunks de ~500 tokens)
    |
    v
Embeddings       (convierte cada chunk en un vector)
    |
    v
Vector Store     (almacena los vectores: FAISS, Chroma, Pinecone)
    |
    v
Retriever        (interfaz de busqueda)


FASE 2: CONSULTA (se ejecuta por cada pregunta del usuario)
------------------------------------------------------------------
Pregunta del usuario
    |
    v
Retriever        (busca los k documentos mas relevantes)
    |
    v
Documentos relevantes
    |
    v
PromptTemplate   (estructura el prompt con contexto + pregunta)
    |
    v
LLM              (genera la respuesta)
    |
    v
OutputParser     (extrae el texto de la respuesta)
    |
    v
Respuesta final
```

**Por que separar las fases** (Lecture M3L2 - Seccion 19.1):

> "No vectorizar documentos en cada request."
> -- Lecture M3L2, Seccion 19.1

En el script legacy, la ingestion ocurria en cada consulta.
Con el pipeline modular, la ingestion se hace UNA vez y el vector store se persiste.


## Bloque 1 - Sin LangChain: el script mezclado

Este es el tipo de script que la lecture describe en la Seccion 3.1.
Todo en una sola funcion, sin retrieval real, sin modularidad.

Ejecuta la celda y fijate en los comentarios.


In [ ]:
from openai import OpenAI

client = OpenAI()

# Datos de la empresa (en produccion vendrian de archivos)
DOCS_EMPRESA = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]


def answer_script_legacy(question: str) -> str:
    """
    Script legacy: todo mezclado.
    Problemas:
    - Usa TODO el contexto siempre (no hay retrieval real)
    - Ingestion y consulta mezcladas
    - Prompt hardcodeado como string
    - Modelo hardcodeado en la funcion
    - No hay forma de debuggear cada paso por separado
    """
    # Problema 1: toma TODO el contexto, no busca el relevante
    context = "\n".join(DOCS_EMPRESA)

    # Problema 2: prompt construido como string manual
    prompt_text = (
        "Eres un asistente de RRHH. Responde usando solo el contexto dado.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Pregunta:\n{question}"
    )

    # Problema 3: llamada directa a OpenAI hardcodeada
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0,
    )

    return response.choices[0].message.content


# Ejecutar el script legacy
respuesta_legacy = answer_script_legacy("Cuantos dias de vacaciones tengo?")
print("Respuesta del script legacy:")
print(respuesta_legacy)
print()
print("--- Problemas ---")
print("1. No hay retrieval: manda los 5 docs aunque la pregunta solo necesita 1")
print("2. No puedo cambiar el modelo sin tocar la funcion")
print("3. No puedo verificar cuales docs se usaron")
print("4. Si el contexto crece (100 docs), el costo de tokens escala mal")


### Por que el retrieval importa (Lecture - Seccion 13.3)

La lecture dice:

> "Si el retrieval esta encapsulado, puedes cambiar la tecnologia sin romper el resto."

Pero hay algo mas importante: con retrieval real, solo mandamos al modelo
los documentos **relevantes** para la pregunta. El script legacy manda todo siempre.

Con 5 documentos no se nota. Con 500 documentos, el costo de tokens puede ser 100x mayor.

Y la calidad de la respuesta puede ser peor: el modelo tiene mas ruido para filtrar.


## Bloque 2 - Con LangChain: ingestion modular

Primero construimos el vector store. Esta es la fase de **ingestion**:
se hace una vez y se puede persistir.


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Los mismos textos del script legacy
TEXTOS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]

# Componentes base
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()

print("Componentes base creados.")
print(f"LLM: {llm.model_name}")
print(f"Embeddings: {type(embeddings).__name__}")


### TODO 1: crear el vector store

Usa `FAISS.from_texts(textos, embeddings)` para crear el vector store en memoria.

Esto reemplaza toda la logica de `embed()` y `save_vectors()` del script legacy.
En produccion, este vector store se persistiria en disco.


In [ ]:
# TODO 1: crear el vector store con FAISS.from_texts(TEXTOS, embeddings)
# vectorstore = ...

vectorstore = None  # reemplazar
print(f"Vector store: {type(vectorstore).__name__ if vectorstore else 'TODO no completado'}")


### TODO 2: crear el retriever

El retriever es la interfaz de busqueda. Encapsula el vector store y expone un metodo `.invoke(query)`.

Parametro `k=2`: traer los 2 documentos mas relevantes para la pregunta.

Usa `vectorstore.as_retriever(search_kwargs={"k": 2})`.


In [ ]:
# TODO 2: crear el retriever con search_kwargs={"k": 2}
# retriever = ...

retriever = None  # reemplazar
print(f"Retriever: {type(retriever).__name__ if retriever else 'TODO no completado'}")

# Probar el retriever directamente (esto NO era posible en el script legacy)
if retriever:
    docs_test = retriever.invoke("vacaciones")
    print(f"Documentos recuperados para 'vacaciones': {len(docs_test)}")
    for i, doc in enumerate(docs_test):
        print(f"  Doc {i+1}: {doc.page_content}")


### Por que aislar el retriever (Lecture - Seccion 13.3)

Con el retriever como objeto separado:

- podemos **debuggear** que documentos trae antes de enviarlos al modelo,
- podemos **reemplazar** FAISS por Chroma o Pinecone sin cambiar el resto del pipeline,
- podemos **testear** la calidad del retrieval de forma independiente.

En el script legacy, el "retrieval" era `" ".join(DOCS)`. No habia forma de saber
si los documentos recuperados eran relevantes sin inspeccionar el codigo completo.


### TODO 3: crear el prompt RAG

El prompt debe tener las variables `{context}` y `{question}`.
El sistema debe indicar que responda **solo** con el contexto dado.


In [ ]:
# TODO 3: crear el ChatPromptTemplate con variables {context} y {question}
# rag_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     ("human", "Contexto:\n{context}\n\nPregunta:\n{question}")
# ])

rag_prompt = None  # reemplazar
print(f"Prompt RAG: {type(rag_prompt).__name__ if rag_prompt else 'TODO no completado'}")


### Funcion auxiliar: formatear documentos

Esta funcion convierte la lista de documentos del retriever en un string
que podemos pasar como `{context}` al prompt.


In [ ]:
def format_docs(docs) -> str:
    """Convierte una lista de documentos en un string para el contexto del prompt."""
    return "\n\n".join(doc.page_content for doc in docs)


# Probar la funcion
if retriever:
    docs_ejemplo = retriever.invoke("vacaciones")
    contexto_formateado = format_docs(docs_ejemplo)
    print("Contexto formateado:")
    print(contexto_formateado)


### TODO 4: componer la RAG chain con LCEL

Esta es la composicion completa del pipeline RAG (Lecture - Seccion 16.5):

```python
rag_chain = (
    {
        "context": retriever | format_docs,  # retriever busca, format_docs formatea
        "question": RunnablePassthrough()    # la pregunta pasa tal cual al prompt
    }
    | rag_prompt
    | llm
    | parser
)
```

El diccionario `{"context": ..., "question": ...}` es como LCEL pasa
multiples inputs al `rag_prompt`. `RunnablePassthrough()` significa
"pasar el input original sin modificarlo".


In [ ]:
# TODO 4: componer la RAG chain completa con LCEL
# rag_chain = (
#     {
#         "context": retriever | format_docs,
#         "question": RunnablePassthrough()
#     }
#     | rag_prompt
#     | llm
#     | parser
# )

rag_chain = None  # reemplazar
print(f"RAG chain: {type(rag_chain).__name__ if rag_chain else 'TODO no completado'}")


### TODO 5: invocar la RAG chain

La RAG chain recibe solo la pregunta como string. Internamente:
1. El retriever busca los documentos relevantes.
2. `format_docs` los convierte en texto.
3. El prompt arma el mensaje con contexto y pregunta.
4. El LLM genera la respuesta.
5. El parser extrae el texto.


In [ ]:
# TODO 5: invocar la rag_chain con esta pregunta
# respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
# print(f"Respuesta RAG: {respuesta}")


## Framework de debugging del pipeline RAG (Lecture M3L2 - Seccion 18.1)

Cuando la respuesta es mala, la lecture propone este proceso:

```text
Respuesta mala
    |
    v
1. El retriever recupero documentos correctos?
   -> retriever.invoke(question)
   -> ver que documentos trajo
    |
    v
2. El contexto era suficiente?
   -> format_docs(docs)
   -> leer el texto que fue al prompt
    |
    v
3. El prompt estaba bien formado?
   -> rag_prompt.format_messages(context=..., question=...)
   -> ver el mensaje exacto que recibio el modelo
    |
    v
4. El modelo recibio las variables correctas?
   -> revisar mensajes del paso 3
    |
    v
5. El parser proceso bien la salida?
   -> verificar tipo de la respuesta (debe ser str)
```

**Diferencia con el script legacy**: en el script legacy, si la respuesta era mala,
no habia forma de saber en que paso fallo. Solo habia un print del resultado final.

Con el pipeline modular, CADA componente es inspectable por separado.


## Bloque 3 - El poder del debugging modular

Con el pipeline modular podemos inspeccionar cada componente por separado.
Esto es imposible con el script legacy.


In [ ]:
# Debugging modular: inspeccion de cada componente
if retriever and rag_prompt:
    question = "Cuantos dias de vacaciones tengo?"

    print("=== Inspeccion del pipeline ===")
    print()

    # Paso 1: ver que documentos recupero el retriever
    print("[Paso 1] Documentos recuperados por el retriever:")
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs):
        print(f"  Doc {i+1}: {doc.page_content}")
    print()

    # Paso 2: ver como queda el contexto formateado
    print("[Paso 2] Contexto formateado:")
    context = format_docs(docs)
    print(context)
    print()

    # Paso 3: ver el prompt final antes de enviarlo al modelo
    print("[Paso 3] Prompt final enviado al modelo:")
    messages = rag_prompt.format_messages(context=context, question=question)
    for msg in messages:
        print(f"  [{msg.type.upper()}] {msg.content[:80]}...")
    print()

    print("--- Esto NO era posible con el script legacy ---")
    print("Con el pipeline modular podemos debuggear cada etapa por separado.")
    print("Si la respuesta es mala, sabemos en que paso buscar el problema.")


### El framework de debugging (Lecture - Seccion 18.1)

La lecture propone estas preguntas cuando la respuesta es mala:

1. El retriever recupero documentos correctos? → `retriever.invoke(question)`
2. El contexto era suficiente? → `format_docs(docs)`
3. El prompt estaba bien formado? → `rag_prompt.format_messages(...)`
4. El modelo recibio las variables correctas? → ver mensajes del punto 3
5. El parser proceso bien la salida? → verificar tipo de respuesta

Con un script legacy, estas preguntas no tienen respuesta clara.
Con el pipeline modular, cada pregunta tiene una celda de debugging.


## Bloque 4 - Checks automaticos


In [ ]:
def run_checks():
    assert vectorstore is not None, "TODO 1: vectorstore es None"
    assert retriever is not None, "TODO 2: retriever es None"
    assert rag_prompt is not None, "TODO 3: rag_prompt es None"
    assert rag_chain is not None, "TODO 4: rag_chain es None"

    # El retriever devuelve documentos
    docs = retriever.invoke("vacaciones")
    assert len(docs) > 0, "El retriever debe devolver al menos un documento"
    assert len(docs) <= 2, "Con k=2 el retriever no debe devolver mas de 2 documentos"

    # El retriever es selectivo: para 'vacaciones' debe traer el doc relevante
    contenidos = [doc.page_content for doc in docs]
    assert any("vacaciones" in c.lower() or "15 dias" in c.lower() for c in contenidos), \
        "El retriever debe traer el documento sobre vacaciones"

    # La chain devuelve una respuesta
    respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
    assert isinstance(respuesta, str), "La respuesta debe ser un string"
    assert len(respuesta) > 0, "La respuesta no debe estar vacia"
    assert "15" in respuesta, "La respuesta debe mencionar los 15 dias"

    # Verificar que el retriever es selectivo (no trae todos los docs)
    docs_remoto = retriever.invoke("trabajo remoto")
    contenidos_remoto = [doc.page_content for doc in docs_remoto]
    assert any("remoto" in c.lower() for c in contenidos_remoto), \
        "Para 'trabajo remoto' el retriever debe traer el doc relevante"

    print("M3L2 E02 Starter checks passed")


run_checks()


## Cierre - Script legacy vs Pipeline RAG modular

| Aspecto | Script legacy | Pipeline RAG con LangChain |
|---|---|---|
| Retrieval | Manda todos los docs siempre | Solo los k mas relevantes |
| Ingestion | Se repite en cada consulta | Se hace una vez, se persiste |
| Prompt | F-string hardcodeado | ChatPromptTemplate reutilizable |
| Modelo | Hardcodeado en la funcion | Objeto reemplazable |
| Debugging | Opaco: no se puede inspeccionar | Cada paso es inspeccionable |
| Cambiar vector store | Reescribir la funcion | Reemplazar el retriever |
| Cambiar modelo | Buscar en todo el codigo | Cambiar la variable `llm` |
| Testing | Dificil: todo esta acoplado | Cada componente se testea solo |

### Proximo paso: E10 (opcional)

E10 propone el ejercicio de refactorizar un script caotico completo
(el mismo ejercicio del taller grupal del PPTX).
